In [1]:
import cv2
import os
import math

In [8]:
video_paths = {
    "gopro": {'path':"../gopro/g_joanna.MP4",
    'start_offset': 30 ,
    "end_offset" : 5
    }
}

In [9]:
video_paths

{'gopro': {'path': '../gopro/g_joanna.MP4',
  'start_offset': 30,
  'end_offset': 5}}

In [ ]:
#  for each video ( participation ) we need to manually enter start and enf offset 
# ( base on secound ( not important time))

In [ ]:



def setup_video_capture_with_end(config):
    info = {}
    for name, data in config.items():
        cap = cv2.VideoCapture(data["path"])
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        total_duration = total_frames / fps

        actual_end_time = total_duration - data["end_offset"]
        usable_duration = actual_end_time - data["start_offset"]

        info[name] = {
            "cap": cap,
            "fps": fps,
            "start_offset": data["start_offset"],
            "usable_duration": usable_duration
        }
    return info

**Context:  check to confirm the video path and crop boundaries are formatted correctly.**

In [13]:
info = setup_video_capture_with_end(video_paths)
info

{'gopro': {'cap': < cv2.VideoCapture 0x7f3686077910>,
  'fps': 25.0,
  'start_offset': 30,
  'usable_duration': 310.84}}

In [81]:

def process_chunks(video_info, chunk_duration=3):
    # List to store the final results of all chunks
    all_chunks_results = []

    min_usable_duration = min(v["usable_duration"] for v in video_info.values())
    total_chunks = math.floor(min_usable_duration / chunk_duration)

    print(f"Starting processing for {total_chunks} chunks...")

    for chunk_idx in range(total_chunks):
        exp_time_start = chunk_idx * chunk_duration
        exp_time_end = (chunk_idx + 1) * chunk_duration

        chunk_record = {
            "chunk_id": chunk_idx,
            "time": f"{exp_time_start}-{exp_time_end}",
            "cameras_data": {},
            "objects": [],
            "action_probs": {}
        }

        for cam_name, data in video_info.items():
            target_frame = int((data["start_offset"] + exp_time_start) * data["fps"])
            data["cap"].set(cv2.CAP_PROP_POS_FRAMES, target_frame)

            ret, frame = data["cap"].read()
            if ret:
                chunk_record["cameras_data"][cam_name] = {
                    "processed_frame": target_frame,
                    "status": "success"
                }
            else:
                chunk_record["cameras_data"][cam_name] = {"status": "failed"}

        # Add this chunk record to the main list
        all_chunks_results.append(chunk_record)

        if chunk_idx % 20 == 0:
            print(f"Chunk {chunk_idx} added to pipeline.")

    print("Processing complete.")
    return all_chunks_results

**Slices video into synchronized 3-second blocks and extracts a single snapshot frame from the start of each block.**
**Output A list of chunk dictionaries containing timing metadata, frame indices, and empty placeholders for future AI results.**


In [40]:
# cv2 format to the remove the offsets and turn into to 3 scound chunk 

In [82]:
processed_data = process_chunks(info)

Starting processing for 103 chunks...
Chunk 0 added to pipeline.
Chunk 20 added to pipeline.
Chunk 40 added to pipeline.
Chunk 60 added to pipeline.
Chunk 80 added to pipeline.
Chunk 100 added to pipeline.
Processing complete.


In [83]:
processed_data[:3]

[{'chunk_id': 0,
  'time': '0-3',
  'cameras_data': {'gopro': {'processed_frame': 750, 'status': 'success'}},
  'objects': [],
  'action_probs': {}},
 {'chunk_id': 1,
  'time': '3-6',
  'cameras_data': {'gopro': {'processed_frame': 825, 'status': 'success'}},
  'objects': [],
  'action_probs': {}},
 {'chunk_id': 2,
  'time': '6-9',
  'cameras_data': {'gopro': {'processed_frame': 900, 'status': 'success'}},
  'objects': [],
  'action_probs': {}}]

In [77]:
import os
import json
def save_load(path:str,save=False,config = None):
    path_file = './' + path + '.json'
    processed_data = {}

    # path_file = "/content/drive/MyDrive/data/" + path + '.json'
    

    if save:
        
        with open(path_file,'w',encoding='utf-8') as f :
            json.dump(config,f,ensure_ascii=False, indent=4)
    else:
        if os.path.exists(path_file):
            with open(path_file, 'r', encoding='utf-8') as f :
                processed_data = json.load(f)
                return processed_data



**Save and Load Json object**

In [25]:
!pip install ultralytics --quiet

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: Could not install packages due to an OSError: [Errno 16] Device or resource busy: '.nfs00000000055c22b400000737'



In [26]:
from ultralytics import YOLO
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = YOLO('yolov8s-world.pt').to(device)

**install YOLO for obj detection**

In [27]:
custom_classes = [
    "measuring tape", "tape measure", "yellow tape measure", "ruler", # برای متر
    "paper shredder", "shredder machine", "trash can",'shredder' # برای کاغذخردکن
    "scissors", "paper", "pen", "pencil", "marker", # ابزار نوشتاری
    "cardboard box", "carton", "box", "package", # انواع جعبه
    "keyboard", "monitor", "laptop","yellow measuring tape", "measuring tape", "tape measure", 
    "scissors", "metal scissors", "shears", 
    "paper shredder", "shredder machine",
    "keyboard", "computer monitor", "pen", "paper", "box"
]

model.set_classes(custom_classes)

**set custom classes for our recoding , need to later set sepecif set of class for each work**

In [44]:

def process_objects_gopro(processed_data, video_info, model, conf_val=0.12):
    for chunk in processed_data:
        detected_names = []
        
        cam_name = "gopro"
        if cam_name in chunk["cameras_data"] and chunk["cameras_data"][cam_name]["status"] == "success":
            
            meta = chunk["cameras_data"][cam_name]
            cap = video_info[cam_name]["cap"]
            cap.set(cv2.CAP_PROP_POS_FRAMES, meta["processed_frame"])
            ret, frame = cap.read()
            
            if ret:

                results = model.predict(frame, conf=conf_val, device=device, verbose=False)
                
                for r in results:
                    for c in r.boxes.cls:
                        obj_name = custom_classes[int(c)]
                        detected_names.append(obj_name)
        
        # Save to the active objects list (removes duplicates using set)
        
        chunk["objects"] = list(set(detected_names))
        
        if chunk["chunk_id"] % 20 == 0:
            print(f"✅ Chunk {chunk['chunk_id']} analyzed.")
            
    return processed_data

**Runs YOLO object detection on the extracted GoPro frame for each 3-second chunk.   Seeks to the target frame, reads it via OpenCV, predicts objects using the YOLO model (with a default confidence threshold of 0.12), maps numerical predictions back to custom class names, and saves a deduplicated list of detected item**

In [84]:
%%time
processed_data_ob_de = process_objects_gopro(processed_data,info,model)

✅ Chunk 0 analyzed.
✅ Chunk 20 analyzed.
✅ Chunk 40 analyzed.
✅ Chunk 60 analyzed.
✅ Chunk 80 analyzed.
✅ Chunk 100 analyzed.
CPU times: user 5min 16s, sys: 922 ms, total: 5min 16s
Wall time: 1min 1s


**use YOLO with custom class to analyis only gopro cam to detect object and save on that same structure**


In [85]:
processed_data_ob_de[:3]

[{'chunk_id': 0,
  'time': '0-3',
  'cameras_data': {'gopro': {'processed_frame': 750, 'status': 'success'}},
  'objects': ['monitor', 'computer monitor', 'keyboard'],
  'action_probs': {}},
 {'chunk_id': 1,
  'time': '3-6',
  'cameras_data': {'gopro': {'processed_frame': 825, 'status': 'success'}},
  'objects': ['monitor', 'computer monitor', 'keyboard'],
  'action_probs': {}},
 {'chunk_id': 2,
  'time': '6-9',
  'cameras_data': {'gopro': {'processed_frame': 900, 'status': 'success'}},
  'objects': ['monitor', 'computer monitor', 'keyboard'],
  'action_probs': {}}]

In [86]:
def normalize_chunk_data(processed_data):
    for chunk in processed_data:
        raw_list = chunk.get("objects", [])
        normalized = set()
        
        for item in raw_list:
            item = item.lower()
            # منطق مپ کردن کلمات
            if any(x in item for x in ["tape", "measuring", "ruler"]):
                normalized.add("MEASURING_TAPE")
            elif any(x in item for x in ["scissors", "shears", "metal"]):
                normalized.add("SCISSORS")
            elif "shredder" in item:
                normalized.add("SHREDDER")
            elif "keyboard" in item:
                normalized.add("KEYBOARD")
            elif "monitor" in item or "screen" in item:
                normalized.add("MONITOR")
            elif "pen" in item or "marker" in item:
                normalized.add("PEN")
            elif "paper" in item:
                normalized.add("PAPER")
            elif "box" in item or "cardboard" in item:
                normalized.add("BOX")
            else:
                normalized.add(item.upper())
        
        chunk["objects"] = list(normalized)
    return processed_data

**the layer normoliser custoom class to one things**

In [87]:
norm_data = normalize_chunk_data(processed_data_ob_de)

In [96]:
import copy 
norm_data_deep_copy = copy.deepcopy(norm_data)

In [97]:
norm_data_deep_copy[30:34]

[{'chunk_id': 30,
  'time': '90-93',
  'cameras_data': {'gopro': {'processed_frame': 3000, 'status': 'success'}},
  'objects': ['PEN', 'KEYBOARD'],
  'action_probs': {},
  'hand_motion': 'REACH_OBJECT'},
 {'chunk_id': 31,
  'time': '93-96',
  'cameras_data': {'gopro': {'processed_frame': 3075, 'status': 'success'}},
  'objects': ['PEN', 'KEYBOARD'],
  'action_probs': {},
  'hand_motion': 'REACH_OBJECT'},
 {'chunk_id': 32,
  'time': '96-99',
  'cameras_data': {'gopro': {'processed_frame': 3150, 'status': 'success'}},
  'objects': ['PEN', 'KEYBOARD', 'LAPTOP'],
  'action_probs': {},
  'hand_motion': 'REACH_OBJECT'},
 {'chunk_id': 33,
  'time': '99-102',
  'cameras_data': {'gopro': {'processed_frame': 3225, 'status': 'success'}},
  'objects': ['LAPTOP', 'KEYBOARD'],
  'action_probs': {},
  'hand_motion': 'IDLE'}]

In [89]:
save_load(path='norm_data2_obj',save=True,config=norm_data_deep_copy)

###  the part in blow are using google medipipe for hand action ( reach object or grasp object or IDEL or active ) 
###  after review output and see what use this code section in first approch we dicede dont use on offical out put 
### because at first we want to use this  hand action and object detect to predict acitvity but later the approch
###  get change and we dont need this 

In [53]:
!pip install mediapipe --quiet

In [54]:
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
!pip install "numpy<2"
!pip install --upgrade matplotlib --quiet

Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-1.26.4-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: Could not install packages due to an OSError: [Errno 16] Device or resource busy: '.nfs00000000044024590000073a'



In [55]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np

base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,
    min_hand_detection_confidence=0.5
)
detector = vision.HandLandmarker.create_from_options(options)

I0000 00:00:1781804978.188196 3966137 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1781804978.268785 3966189 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 595.71.05), renderer: NVIDIA A2/PCIe/SSE2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1781804978.309833 3966143 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781804978.345016 3966148 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [56]:

def get_hand_action(frame, detector):
    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
    result = detector.detect(mp_image)

    if not result.hand_landmarks:
        return "IDLE"

    for landmarks in result.hand_landmarks:
        # Index IDs for finger tips and their corresponding PIP joints
        tips_ids = [8, 12, 16, 20]  # Index, Middle, Ring, Pinky
        pip_ids = [6, 10, 14, 18]
        
        open_fingers = 0
        
        # Check the 4 fingers (excluding the thumb)
        for tip, pip in zip(tips_ids, pip_ids):
            # In MediaPipe coordinates, the y-value increases from top to bottom.
            # If the tip's y is less than the joint's y, the finger is extended (open).
            if landmarks[tip].y < landmarks[pip].y:
                open_fingers += 1
                
        if open_fingers >= 3:
            return "REACH_OBJECT"  
        elif open_fingers <= 1:
            return "GRASP_OBJECT"  
        else:
            return "ACTIVE"      

    return "IDLE"

In [90]:
import cv2

def update_json_with_smart_fusion(processed_data, video_info, detector):
    processed_data_copy = processed_data.copy()
    for chunk in processed_data_copy:
        motions = []
        for cam, meta in chunk["cameras_data"].items():
            if meta["status"] == "success":
                cap = video_info[cam]["cap"]
                cap.set(cv2.CAP_PROP_POS_FRAMES, meta["processed_frame"])
                ret, frame = cap.read()
                if ret:
                    # Call the hand action detector function
                    motions.append(get_hand_action(frame, detector))
        
        # If no motion data was captured from any camera, default to IDLE
        if not motions:
            chunk["hand_motion"] = "IDLE"
        else:
            # Majority voting: select the most frequent action among all cameras
            most_common = max(set(motions), key=motions.count)
            chunk["hand_motion"] = most_common
            
    return processed_data

In [91]:
processed_data_ob_de_ha = update_json_with_smart_fusion(processed_data_ob_de,info,detector)

In [92]:
processed_data_ob_de_ha[42:49]

[{'chunk_id': 42,
  'time': '126-129',
  'cameras_data': {'gopro': {'processed_frame': 3900, 'status': 'success'}},
  'objects': ['PEN', 'KEYBOARD', 'LAPTOP'],
  'action_probs': {},
  'hand_motion': 'GRASP_OBJECT'},
 {'chunk_id': 43,
  'time': '129-132',
  'cameras_data': {'gopro': {'processed_frame': 3975, 'status': 'success'}},
  'objects': ['LAPTOP', 'SCISSORS', 'KEYBOARD', 'PEN'],
  'action_probs': {},
  'hand_motion': 'GRASP_OBJECT'},
 {'chunk_id': 44,
  'time': '132-135',
  'cameras_data': {'gopro': {'processed_frame': 4050, 'status': 'success'}},
  'objects': ['PEN', 'KEYBOARD'],
  'action_probs': {},
  'hand_motion': 'ACTIVE'},
 {'chunk_id': 45,
  'time': '135-138',
  'cameras_data': {'gopro': {'processed_frame': 4125, 'status': 'success'}},
  'objects': ['PEN', 'KEYBOARD'],
  'action_probs': {},
  'hand_motion': 'REACH_OBJECT'},
 {'chunk_id': 46,
  'time': '138-141',
  'cameras_data': {'gopro': {'processed_frame': 4200, 'status': 'success'}},
  'objects': ['PEN', 'KEYBOARD'],
